# Notebook 04b：SE(3)、Twist、Wrench 与 Adjoint

## 1. 本节在知识体系中的位置

```
NB04 四元数/SO(3) ──→ NB04b SE(3) ──→ NB05b PoE FK
                           │
                           ├── twist/wrench 的统一表达
                           └── Adjoint 变换（力/速度在不同系之间的映射）
```

SO(3) 只描述旋转。SE(3) 统一描述旋转+平移——这是机器人学中刚体运动的完整数学框架。

## 2. 学习目标

- ⭐ 掌握 hat (∧) 和 vee (∨) 操作符
- ⭐ 理解 twist $\mathcal{V} = [\mathbf{v}; \boldsymbol{\omega}]$ 的物理含义
- ⭐ 区分 space twist 和 body twist
- ⭐ 掌握 SE(3) 指数映射和 twist 的物理意义
- ⭐ 掌握 Adjoint 变换
- ⭐ 理解 wrench 的对偶变换
- 📖 区分 left/right perturbation

## 3. 约定

本课程使用 **Lynch & Park (Modern Robotics)** 约定：
- twist 排列：$\mathcal{V} = [\mathbf{v}; \boldsymbol{\omega}]$（线速度在前，角速度在后）
- hat 操作符：$\boldsymbol{\xi}^\wedge = \begin{bmatrix} [\boldsymbol{\omega}]_\times & \mathbf{v} \\ \mathbf{0}^T & 0 \end{bmatrix} \in \mathfrak{se}(3)$
- 空间 twist $\mathcal{V}_s$ 和物体 twist $\mathcal{V}_b$ 的关系：$\mathcal{V}_s = \text{Ad}_{T} \mathcal{V}_b$

## 4. Hat 和 Vee ⭐

### 4.1 $\mathfrak{so}(3)$ 上的 hat/vee

$$\boldsymbol{\omega}^\wedge = [\boldsymbol{\omega}]_\times = \begin{bmatrix} 0 & -\omega_z & \omega_y \\ \omega_z & 0 & -\omega_x \\ -\omega_y & \omega_x & 0 \end{bmatrix} \in \mathfrak{so}(3)$$
$$([\boldsymbol{\omega}]_\times)^\vee = \boldsymbol{\omega}$$

### 4.2 $\mathfrak{se}(3)$ 上的 hat/vee

$$\boldsymbol{\xi}^\wedge = \begin{bmatrix} \mathbf{v} \\ \boldsymbol{\omega} \end{bmatrix}^\wedge = \begin{bmatrix} [\boldsymbol{\omega}]_\times & \mathbf{v} \\ \mathbf{0}^T & 0 \end{bmatrix} \in \mathfrak{se}(3)$$
$$\left(\begin{bmatrix} [\boldsymbol{\omega}]_\times & \mathbf{v} \\ \mathbf{0}^T & 0 \end{bmatrix}\right)^\vee = \begin{bmatrix} \mathbf{v} \\ \boldsymbol{\omega} \end{bmatrix}$$

## 5. Twist（旋量）⭐

### 5.1 空间 Twist（Spatial Twist）

刚体运动可视为绕空间系中某个螺旋轴 $\mathcal{S}$ 的旋转+平移。瞬时速度用 twist 描述：
$$\mathcal{V}_s = \begin{bmatrix} \mathbf{v}_s \\ \boldsymbol{\omega}_s \end{bmatrix} \in \mathbb{R}^6$$

$\mathbf{v}_s$ 不是物体上某点的线速度，而是**假设物体无限延伸、在空间系原点处**的线速度。

空间 twist 与 SE(3) 的关系：
$$\dot{T} = \mathcal{V}_s^\wedge T$$

### 5.2 物体 Twist（Body Twist）

物体 twist 是速度在**物体自身参考系**中的表达：
$$\mathcal{V}_b = \begin{bmatrix} \mathbf{v}_b \\ \boldsymbol{\omega}_b \end{bmatrix} = \text{Ad}_{T^{-1}} \mathcal{V}_s$$

物体 twist 与 SE(3) 的关系：
$$\dot{T} = T \mathcal{V}_b^\wedge$$

### 5.3 从 Twist 恢复 SE(3) 的指数映射

给定空间 twist $\mathcal{V}_s = [\mathbf{v}_s; \boldsymbol{\omega}_s]$，在时间 $\Delta t$ 后的位姿变化：
$$T(\Delta t) = \exp(\mathcal{V}_s^\wedge \Delta t) T(0)$$

其中 SE(3) 指数映射已经实现在 `se3_exp()` 中（见 NB04）。

## 6. Adjoint 变换 ⭐

### 6.1 Adjoint 矩阵

给定 $T = (R, \mathbf{p}) \in SE(3)$，Adjoint 矩阵 $\text{Ad}_T \in \mathbb{R}^{6\times 6}$ 将物体系中的 twist 变换到空间系：
$$\mathcal{V}_s = \text{Ad}_T \mathcal{V}_b$$

$$\text{Ad}_T = \begin{bmatrix} R & [\mathbf{p}]_\times R \\ 0 & R \end{bmatrix}$$

### 6.2 Wrench 的对偶变换

Wrench $\mathcal{F} = [\mathbf{f}; \mathbf{n}]$（力+力矩）按照 Adjoint 的**转置逆**变换：
$$\mathcal{F}_s = \text{Ad}_T^{-T} \mathcal{F}_b = \begin{bmatrix} R & 0 \\ [\mathbf{p}]_\times R & R \end{bmatrix} \begin{bmatrix} \mathbf{f}_b \\ \mathbf{n}_b \end{bmatrix}$$

这解释了为什么 $\boldsymbol{\tau} = \mathbf{J}^T \mathbf{F}$——雅可比转置本质上是在做 wrench 的 Adjoint 变换。

## 7. Spatial Jacobian vs Body Jacobian

空间雅可比 $\mathbf{J}_s(\mathbf{q})$ 和物体雅可比 $\mathbf{J}_b(\mathbf{q})$ 的关系：
$$\mathbf{J}_s(\mathbf{q}) = \text{Ad}_{T_{sb}(\mathbf{q})} \mathbf{J}_b(\mathbf{q})$$

- $\mathbf{J}_s$ 的每一列是关节 $i$ 的 twist 在**空间系**中的表达
- $\mathbf{J}_b$ 的每一列是关节 $i$ 的 twist 在**物体系**中的表达
- NB07 的 `compute_geometric_jacobian` 计算的是**空间雅可比**

## 8. Python 实现

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '..')
from src.robotics_learning.transforms import skew, so3_exp, axis_angle_to_rot, rot_z, homogenous_transform
%matplotlib inline
rng = np.random.RandomState(42)
print("✅ 导入完成")

### 8.1 Hat/Vee 和 SE(3) 工具

In [ ]:
def se3_hat(twist):
    """ξ = [v; ω] → ξ^∧ ∈ se(3)"""
    v, omega = twist[:3], twist[3:]
    Xi = np.zeros((4, 4))
    Xi[:3, :3] = skew(omega)
    Xi[:3, 3] = v
    return Xi

def se3_vee(Xi):
    """ξ^∧ ∈ se(3) → ξ = [v; ω]"""
    omega = np.array([Xi[2,1], Xi[0,2], Xi[1,0]])
    v = Xi[:3, 3]
    return np.concatenate([v, omega])

def adjoint(T):
    """T ∈ SE(3) → Ad_T ∈ R^{6×6}"""
    R = T[:3, :3]; p = T[:3, 3]
    Ad = np.zeros((6, 6))
    Ad[:3, :3] = R; Ad[3:, 3:] = R
    Ad[:3, 3:] = skew(p) @ R
    return Ad

def adjoint_inv_transpose(T):
    """T → Ad_T^{-T} (wrench 变换)"""
    R = T[:3, :3]; p = T[:3, 3]
    Ad_invT = np.zeros((6, 6))
    Ad_invT[:3, :3] = R
    Ad_invT[3:, :3] = skew(p) @ R
    Ad_invT[3:, 3:] = R
    return Ad_invT

# 测试: 随机 SE(3) 的 Adjoint 往返
R_test = axis_angle_to_rot(rng.randn(3), 1.0)
T_test = homogenous_transform(R_test, rng.uniform(-1, 1, 3))
Ad = adjoint(T_test)
# Ad_T · Ad_{T^{-1}} = I
Ad_inv = adjoint(homogenous_transform(R_test.T, -R_test.T @ T_test[:3, 3]))
assert np.allclose(Ad @ Ad_inv, np.eye(6), atol=1e-10)
print("✅ Adjoint 往返测试通过")

### 8.2 Twist 变换演示

In [ ]:
# 2R 臂在 q1=30°, q2=45° 的空间雅可比
from src.robotics_learning.kinematics import compute_geometric_jacobian, forward_kinematics
dh = np.array([[1.0, 0, 0], [0.8, 0, 0]])
q = np.array([np.pi/6, np.pi/4])
J_s = compute_geometric_jacobian(dh, q)

# 末端位姿 T
T_end, _ = forward_kinematics(np.column_stack([dh, q]))
Ad_end = adjoint(T_end)

# 物体雅可比: J_b = Ad_{T^{-1}} J_s
J_b = np.linalg.solve(Ad_end, J_s)  # Ad_T^{-1} · J_s

# 验证: 同一关节速度下，空间 twist 和物体 twist 的关系
q_dot = np.array([1.0, -0.5])
V_s = J_s @ q_dot   # 空间 twist
V_b = J_b @ q_dot   # 物体 twist
V_s_from_adj = Ad_end @ V_b
print(f"V_s = J_s q̇:\n{np.round(V_s, 4)}")
print(f"Ad_T · V_b:\n{np.round(V_s_from_adj, 4)}")
print(f"一致? {np.allclose(V_s, V_s_from_adj, atol=1e-10)}")

### 8.3 Wrench 变换验证

In [ ]:
# 末端受外力 F_b = [fx, fy, fz, nx, ny, nz] (在 body 系)
F_b = np.array([10.0, -5.0, 0.0, 0.0, 0.0, 2.0])
# 转换到空间系
Ad_invT = adjoint_inv_transpose(T_end)
F_s = Ad_invT @ F_b

# 关节力矩: τ = J_s^T F_s = J_b^T F_b
tau_from_space = J_s.T @ F_s
tau_from_body = J_b.T @ F_b
print(f"τ (from space J^T F_s): {np.round(tau_from_space, 4)}")
print(f"τ (from body J^T F_b):   {np.round(tau_from_body, 4)}")
print(f"一致? {np.allclose(tau_from_space, tau_from_body, atol=1e-10)}")

### 8.4 左扰动 vs 右扰动

In [ ]:
# 左扰动: T_new = exp(δξ^∧) · T   (在空间系施加扰动)
# 右扰动: T_new = T · exp(δξ^∧)   (在物体系施加扰动)
delta = np.array([0.01, 0, 0, 0, 0, 0.05])  # 小扰动 [v; ω]
Xi = se3_hat(delta)

T_original = T_end.copy()
T_left = np.real(T_original.copy())
# 简化: T_left ≈ (I + Xi) @ T (一级近似)
T_left = (np.eye(4) + Xi) @ T_left
T_right = T_original @ (np.eye(4) + Xi)

# 检查两种扰动的区别
print(f"左扰动后位置: {np.round(T_left[:3,3], 4)}")
print(f"右扰动后位置: {np.round(T_right[:3,3], 4)}")
print(f"原始位置:      {np.round(T_original[:3,3], 4)}")
print("左扰动：δ 在空间系表达 → 影响物体系的全局位置。")
print("右扰动：δ 在物体系表达 → 影响物体相对于自身的姿态。")

## 9. 常见错误

1. **twist 排列不一致**：有些教材用 $[\boldsymbol{\omega}; \mathbf{v}]$。本课程统一用 $[\mathbf{v}; \boldsymbol{\omega}]$（Lynch & Park 约定）。两个排列下 Adjoint 矩阵的结构不同。
2. **空间 twist vs 物体 twist**：$\mathcal{V}_s$ 中 $\mathbf{v}_s$ 不是物体上某点的实际线速度。真实末端线速度 = $\mathbf{v}_s + \boldsymbol{\omega}_s \times \mathbf{p}$。
3. **Adjoint vs 坐标变换**：$\text{Ad}_T$ 将 twist/wrench 在坐标系间映射，不是简单的 $\mathbb{R}^6$ 旋转。

## 10. 练习题

### 概念题
1. $\mathcal{V}_s$ 和 $\mathcal{V}_b$ 中的线速度分量 $\mathbf{v}_s$ 和 $\mathbf{v}_b$ 分别表示什么意思？
2. 为什么 wrench 的变换是 $\text{Ad}_T^{-T}$ 而非 $\text{Ad}_T$？

### 手算题
1. 给定 $T = (R_z(30°), \mathbf{p}=[1,2,0]^T)$，计算 $\text{Ad}_T$。
2. 验证 $\text{Ad}_{T_1 T_2} = \text{Ad}_{T_1} \text{Ad}_{T_2}$。

### 编程题
1. 验证 $\mathcal{V}_b^T \mathcal{F}_b = \mathcal{V}_s^T \mathcal{F}_s$（功率不变性）。
2. 实现 SE(3) 的 exp 映射并用数值差分验证 $\dot{T} = \mathcal{V}_s^\wedge T$。

> 答案见 `solutions/solutions_week2.ipynb`